In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("transactions.csv", parse_dates=["date"])

In [3]:
df.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [4]:
df.dtypes

date            datetime64[ns]
store_nbr                int64
transactions             int64
dtype: object

In [5]:
df.shape

(83488, 3)

In [6]:
# !pip install pyarrow

In [7]:
df.to_parquet("transactions.parquet", engine="pyarrow", index=False)

In [8]:
pd.read_parquet("transactions.parquet")

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


In [10]:
df.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [11]:
df.dtypes

date            datetime64[ns]
store_nbr                int64
transactions             int64
dtype: object

In [15]:
(
    df.groupby(
        by=["store_nbr"])
    .agg({"transactions": "sum"})
    .sort_values(by="transactions", ascending=False)
    .iloc[:10]
)

,transactions
store_nbr,
44,7273093
47,6535810
45,6201115
46,5990113
3,5366350
48,5107785
8,4637971
49,4574103
50,4384444


In [16]:
df["month"] = df["date"].dt.month

In [17]:
df.head()

,date,store_nbr,transactions,month
0,2013-01-01,25,770,1
1,2013-01-02,1,2111,1
2,2013-01-02,2,2358,1
3,2013-01-02,3,3487,1
4,2013-01-02,4,1922,1


In [21]:
(
    df
    .groupby(by=["store_nbr", "month"])
    .agg({"transactions": "sum"})
    .sort_values(by=["month", "transactions"], ascending=[True, False])
    .iloc[:10]
)


,,transactions
store_nbr,month,
44,1,628438
47,1,568824
45,1,538370
46,1,522763
3,1,463260
48,1,439045
8,1,404463
49,1,386589
50,1,372093


In [22]:
(
    df
    .groupby(by=["store_nbr", "month"])
    .agg(transactions_sum=("transactions", "sum"), transactions_max=("transactions", "max"))
    .sort_values(by=["month", "transactions_sum"], ascending=[True, False])
    .iloc[:10]
)

,,transactions_sum,transactions_max
store_nbr,month,,
44,1,628438,5701
47,1,568824,5234
45,1,538370,5218
46,1,522763,5600
3,1,463260,3920
48,1,439045,4836
8,1,404463,3250
49,1,386589,4014
50,1,372093,3835


In [23]:
(df
.groupby(by=["store_nbr", "month"])
.agg({"transactions": ["sum", "max"]})
.iloc[:10]
)

transactions      
                         sum   max
store_nbr month                   
1         1           229203  2111
          2           209400  2130
          3           232178  1965
          4           229081  2133
          5           231160  2087
          6           231514  1964
          7           236032  2015
          8           202821  1930
          9           182035  1923
          10          184419  1935

In [25]:
(df
.groupby(by=["store_nbr", "month"])
.agg({"transactions": ["sum", "max"]})
.sort_values(["month", ("transactions", "sum")], ascending=[True, False])
.iloc[:10]
)

transactions      
                         sum   max
store_nbr month                   
44        1           628438  5701
47        1           568824  5234
45        1           538370  5218
46        1           522763  5600
3         1           463260  3920
48        1           439045  4836
8         1           404463  3250
49        1           386589  4014
50        1           372093  3835
11        1           336187  3547

In [43]:
df_grouped = (df
              .groupby(by=["store_nbr", "month"])
              .agg({"transactions": ["sum", "mean"]})
              .sort_values(["month", ("transactions", "mean")], ascending=[True, False])
              )

In [44]:
df_grouped.head()

transactions             
                         sum         mean
store_nbr month                          
44        1           628438  4246.202703
47        1           568824  3843.405405
45        1           538370  3637.635135
46        1           522763  3532.182432
3         1           463260  3151.428571

In [45]:
df_grouped.loc[(3, 1), ("transactions", "mean")]

np.float64(3151.4285714285716)

In [46]:
df_grouped.loc[:, ("transactions", "mean")]

store_nbr  month
44         1        4246.202703
47         1        3843.405405
45         1        3637.635135
46         1        3532.182432
3          1        3151.428571
                       ...     
26         12        890.991667
22         12        844.166667
30         12        813.875000
35         12        727.466667
32         12        718.058333
Name: (transactions, mean), Length: 641, dtype: float64

In [47]:
df_grouped.loc[(slice(None), 6), ("transactions", "mean")]

store_nbr  month
44         6        4206.773333
47         6        3750.393333
45         6        3608.680000
46         6        3440.226667
3          6        3164.453333
48         6        2906.766667
8          6        2708.520000
49         6        2677.193333
50         6        2567.920000
11         6        2301.533333
24         6        2267.683333
34         6        2181.840000
52         6        2092.533333
9          6        2046.793333
2          6        1898.900000
6          6        1820.180000
7          6        1814.200000
51         6        1737.606667
38         6        1663.306667
1          6        1543.426667
20         6        1520.866667
4          6        1484.520000
37         6        1482.520000
27         6        1476.986667
39         6        1413.526667
17         6        1388.880000
5          6        1376.006667
14         6        1373.053333
31         6        1353.966667
15         6        1337.986667
18         6        132

In [49]:
df_grouped.xs(6, level="month")[("transactions", "mean")]

store_nbr
44    4206.773333
47    3750.393333
45    3608.680000
46    3440.226667
3     3164.453333
48    2906.766667
8     2708.520000
49    2677.193333
50    2567.920000
11    2301.533333
24    2267.683333
34    2181.840000
52    2092.533333
9     2046.793333
2     1898.900000
6     1820.180000
7     1814.200000
51    1737.606667
38    1663.306667
1     1543.426667
20    1520.866667
4     1484.520000
37    1482.520000
27    1476.986667
39    1413.526667
17    1388.880000
5     1376.006667
14    1373.053333
31    1353.966667
15    1337.986667
18    1322.373333
40    1313.253333
43    1276.214765
12    1228.253333
19    1221.633333
28    1177.946667
42    1128.200000
53    1106.550000
29    1087.877778
36    1061.880000
23    1050.313333
41    1037.920000
33    1033.413333
21    1001.950000
10     979.946309
13     921.233333
16     879.633333
54     861.073826
22     741.150000
25     730.740000
30     712.240000
35     628.308725
32     622.960000
26     622.340000
Name: (transaction

In [50]:
df_grouped.loc

In [51]:
df_grouped.head()

transactions             
                         sum         mean
store_nbr month                          
44        1           628438  4246.202703
47        1           568824  3843.405405
45        1           538370  3637.635135
46        1           522763  3532.182432
3         1           463260  3151.428571

In [53]:
df_grouped.droplevel(0, axis=1).reset_index()

,store_nbr,month,sum,mean
0,44,1,628438,4246.202703
1,47,1,568824,3843.405405
2,45,1,538370,3637.635135
3,46,1,522763,3532.182432
4,3,1,463260,3151.428571
...,...,...,...,...
636,26,12,106919,890.991667
637,22,12,50650,844.166667
638,30,12,97665,813.875000
639,35,12,87296,727.466667


In [55]:
df.head()

,date,store_nbr,transactions,month
0,2013-01-01,25,770,1
1,2013-01-02,1,2111,1
2,2013-01-02,2,2358,1
3,2013-01-02,3,3487,1
4,2013-01-02,4,1922,1


In [76]:

df = df.assign(
    target_pct=df['transactions'] / 2500,
    met_target=df['transactions'] >= 2500,
    bonus_payable=((df['transactions'] / 2500) >= 1) * 100,
    day_of_week=df['date'].dt.dayofweek,
)

In [72]:
df.head()

,date,store_nbr,transactions,month
0,2013-01-01,25,770,1
1,2013-01-02,1,2111,1
2,2013-01-02,2,2358,1
3,2013-01-02,3,3487,1
4,2013-01-02,4,1922,1


In [79]:
(
    df
    .groupby(by=['month'])
    .agg({"met_target": 'mean', "bonus_payable": 'sum'})
    .sort_values('bonus_payable', ascending=False)
)

,met_target,bonus_payable
month,,
12,0.255640,154100
5,0.170792,131800
3,0.169461,130400
4,0.174469,129700
7,0.162486,126300
2,0.174230,121700
6,0.161706,121700
8,0.174189,120800
1,0.163723,119600


In [80]:
(
    df
    .groupby(by=['day_of_week'])
    .agg({"met_target": 'mean', "bonus_payable": 'sum'})
    .sort_values('bonus_payable', ascending=False)
)

,met_target,bonus_payable
day_of_week,,
5,0.222204,266400
6,0.204001,241700
4,0.179007,213000
0,0.160214,191600
2,0.160572,191000
1,0.146299,175500
3,0.142077,169100


In [84]:
(
    df.assign(
        average_store_transactions=df.groupby(["store_nbr", "day_of_week"])["transactions"].transform(
            "mean"),
        difference=lambda x: x["transactions"] - x["average_store_transactions"]
    )
)

,date,store_nbr,transactions,month,target_pct,met_target,bonus_payable,day_of_week,average_store_transactions,difference
0,2013-01-01,25,770,1,0.3080,False,0,1,740.245690,29.754310
1,2013-01-02,1,2111,1,0.8444,False,0,2,1870.782427,240.217573
2,2013-01-02,2,2358,1,0.9432,False,0,2,1952.652720,405.347280
3,2013-01-02,3,3487,1,1.3948,True,100,2,3142.682008,344.317992
4,2013-01-02,4,1922,1,0.7688,False,0,2,1499.569038,422.430962
...,...,...,...,...,...,...,...,...,...,...
83483,2017-08-15,50,2804,8,1.1216,True,100,1,2342.410788,461.589212
83484,2017-08-15,51,1573,8,0.6292,False,0,1,1548.448133,24.551867
83485,2017-08-15,52,2255,8,0.9020,False,0,1,1892.588235,362.411765
83486,2017-08-15,53,932,8,0.3728,False,0,1,877.214286,54.785714


<img src="https://pandas.pydata.org/pandas-docs/stable/_images/reshaping_pivot.png">

<img src="https://pandas.pydata.org/pandas-docs/stable/_images/reshaping_stack.png">